# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,RKLB,20,12,0.118567,2026-05-15 23:10:06+00:00
1,NBIS,15,11,0.461564,2026-05-15 23:10:06+00:00
2,LNG,49,1,0.998300,2026-05-14 11:08:46+00:00
3,AMD,12,9,0.269733,2026-05-15 15:12:08+00:00
4,APLD,33,1,0.999400,2026-05-15 15:10:18+00:00
5,NVDA,9,7,0.722486,2026-05-14 21:05:35+00:00
6,FRMI,25,1,0.999400,2026-05-15 15:10:18+00:00
7,MSFT,8,3,0.175300,2026-05-15 19:00:57+00:00
8,QQQ,4,3,0.446767,2026-05-14 14:16:39+00:00
9,INTC,4,3,0.051033,2026-05-15 16:40:21+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['RKLB', 'NBIS', 'LNG', 'AMD']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
    analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": float(features_df["Close"].dropna().iloc[-1]),
        "latest_rsi": float(features_df["RSI"].dropna().iloc[-1]),
        "latest_macd": float(features_df["MACD"].dropna().iloc[-1]),
        "return_21d": float(features_df["21D Return"].dropna().iloc[-1]),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,AMD,251,424.100006,67.292450,50.243086,0.524114,15.051843,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
1,LNG,251,241.839996,37.718043,-6.133373,-0.079546,4.034770,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
2,NBIS,251,219.940002,72.139888,19.891508,0.330229,7.715465,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
3,RKLB,251,124.769997,70.818158,12.835091,0.504522,4.433056,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== RKLB latest metrics =====
                 Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI       MACD  5D Return  21D Return
Date                                                                                                                                      
2026-05-11  117.349998  54779700           12.303035           78.388000            74.558027  73.376079   6.433867   0.461213    0.724467
2026-05-12  117.559998  34195200           13.597621           80.394000            75.543393  73.450893   8.217018   0.492636    0.664684
2026-05-13  124.150002  26587400           15.419684           82.391667            76.657103  75.753444  10.046130   0.466627    0.719053
2026-05-14  132.550003  25324100           17.593759           84.626000            77.937669  78.332955  12.034796   0.686816    0.800951
2026-05-15  124.769997  22310501           18.749055           86.527333            79.010571  70.818158  12.835091   0.182990    0.5

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== RKLB regression results =====
                 Close  Predicted Close
Date                                   
2026-05-04   80.309998        84.037122
2026-05-05   78.760002        83.721779
2026-05-06   84.650002        86.972348
2026-05-07   78.580002        83.382939
2026-05-08  105.470001        98.503336
2026-05-11  117.349998       106.258091
2026-05-12  117.559998       111.187518
2026-05-13  124.150002       116.262147
2026-05-14  132.550003       125.582539
2026-05-15  124.769997       120.631938

===== NBIS regression results =====
                 Close  Predicted Close
Date                                   
2026-05-04  176.419998       162.252983
2026-05-05  175.919998       167.490844
2026-05-06  195.089996       178.110461
2026-05-07  184.770004       176.800948
2026-05-08  177.050003       173.979159
2026-05-11  186.100006       178.165674
2026-05-12  179.110001       176.782246
2026-05-13  207.270004       187.408462
2026-05-14  221.149994       198.308392
2026-0

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "RKLB",
    "NBIS",
    "LNG",
    "AMD"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
